# **Problem Definition:**

*   Predicting Household Electricity Consumption Using Regression Model **bold text**



# **Data:**

* The Household Electric Power Consumption dataset contains measurements of electricity usage recorded in a household over time.
* It includes several electrical variables such as power consumption, voltage, current intensity, and sub-metering values used to analyze and predict electricity usage.

# **Dataset Description:**

The dataset includes features related to household electrical measurements:

1. Date – Date of the measurement.

2. Time – Time of the measurement.

3. Global_reactive_power – Household reactive power consumption (kW).

4. Voltage – Voltage level (V).

5. Global_intensity – Current intensity (A).

6. Sub_metering_1 – Energy used in kitchen appliances.

7. Sub_metering_2 – Energy used in laundry appliances.

8. Sub_metering_3 – Energy used for water heater and air conditioning.

Target Variable:
* Global_active_power – Total electricity consumption to be predicted.



# ***`1. Import Packages`***

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import scipy.stats as stats
%matplotlib inline

# ***`2. Load Data`***

In [ ]:
from google.colab import files
uploaded = files.upload()

Due to the large size of the dataset, a subset of 200,000 observations was used for analysis and model training to reduce computational complexity while maintaining sufficient data for reliable prediction.

In [ ]:
df = pd.read_csv("power_cosump.csv", sep=",", low_memory=False)

# Take first 200,000 rows for learning
df = df.iloc[:200000].copy()

df.head()

In [ ]:
print(df.columns)

# ***`3. Date-Time Processing`***

In [ ]:
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"])

df["hour"] = df["datetime"].dt.hour
df["day"] = df["datetime"].dt.day
df["month"] = df["datetime"].dt.month

# **`4. Missing Data Analysis`**
If the missing values are not handled properly we may end up drawing an inaccurate inference about the data. Due to improper handling, the result obtained will differ from the ones where missing values are present.

In [ ]:
# Printing total number of missing data
df.isnull().sum().sort_values(ascending=False)

The dataset contained a small number of missing values (61 observations) in several numerical variables. Since the number of missing records was very small compared to the total dataset size, the rows containing missing values were removed to ensure data quality.

In [ ]:
df = df.dropna()

In [ ]:
df.isnull().sum()

# ***`5. Exploratory Data Analysis (EDA)`***
Exploratory Data Analysis (EDA) is performed to understand the structure of the dataset, identify patterns, detect outliers, and analyze relationships between variables before building machine learning models.

# ***`5.1 Pandas Profiling Report`***

In [ ]:
!pip install ydata-profiling
from ydata_profiling import ProfileReport
profile = ProfileReport(df, explorative=True)
profile

# ***`5.2 Distribution Plot `***

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.histplot(df["Global_active_power"], bins=50)
plt.title("Distribution of Global Active Power")
plt.show()

# ***`5.3 Correlation Heatmap`***

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.show()

# ***`6. Handling Outliers`***


> Visualization using Boxplot






In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(15,10))

for i, col in enumerate(df.select_dtypes(include=['float64','int64']).columns):
    plt.subplot(3,3,i+1)
    sns.boxplot(x=df[col])
    plt.title(col)

plt.tight_layout()
plt.show()



> Replacing Outliers with Upper/Lower bound using IQR

Outliers were handled using the Interquartile Range (IQR) method. Upper and lower limits were calculated using the formula
𝑄
1
−
1.5
×
𝐼
𝑄
𝑅
Q1−1.5×IQR and
𝑄
3
+
1.5
×
𝐼
𝑄
𝑅
Q3+1.5×IQR. Values outside these limits were capped to the respective boundary values to reduce the influence of extreme observations while retaining all data points.


In [ ]:
for col in df.select_dtypes(include=['float64','int64']).columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[col] = df[col].clip(lower, upper)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(15,10))

for i, col in enumerate(df.select_dtypes(include=['float64','int64']).columns):
    plt.subplot(3,3,i+1)
    sns.boxplot(x=df[col])
    plt.title(col)

plt.tight_layout()
plt.show()

# ***`7. Skewness Analysis`***
Skewness was calculated for all numerical variables to evaluate the symmetry of their distributions. Variables with high skewness values indicate non-normal distributions and may require transformation to improve model performance.





> Visualization using Histogram



In [ ]:
import matplotlib.pyplot as plt

numeric_df = df.select_dtypes(include=['int64','float64'])

numeric_df.hist(figsize=(15,8), bins=50)

plt.tight_layout()
plt.show()



> Calculating Skewness using skew



In [ ]:
df.skew(numeric_only=True)



Skewness was computed for all numerical variables to assess the symmetry of their distributions. Variables such as Global Active Power, Global Intensity, and Sub-metering 2 exhibited high positive skewness (>1). Therefore, a logarithmic transformation was applied to reduce skewness and stabilize variance.



In [ ]:
import numpy as np

df["Global_active_power"] = np.log1p(df["Global_active_power"])
df["Global_intensity"] = np.log1p(df["Global_intensity"])
df["Sub_metering_2"] = np.log1p(df["Sub_metering_2"])

In [ ]:
df.skew(numeric_only=True)

# ***`8. Multicollinearity Analysis `***
Multicollinearity among predictor variables was examined using the Variance Inflation Factor (VIF). Some electrical variables showed high correlation due to their physical relationship in power systems. To address this issue and reduce redundancy among predictors, Principal Component Analysis (PCA) was later applied during model preprocessing.

> Correlation Plot



In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

> Variance Inflation Factor (VIF)

Variance Inflation Factor (VIF) is used to detect multicollinearity among independent variables.
It measures how much the variance of a regression coefficient increases due to correlation
between predictors.

The VIF is calculated using the formula:

VIF = 1 / (1 - R²)

where R² represents the coefficient of determination obtained when a predictor is
regressed against the remaining predictors.

>  Interpretation of VIF values
- VIF = 1 → No correlation
- 1 < VIF < 5 → Moderate correlation
- VIF > 5 → High multicollinearity

In [ ]:
X = df.drop(columns=["Global_active_power"])
y = df["Global_active_power"]

In [ ]:
import pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Drop non-numeric columns from X before calculating VIF
X_numeric = X.drop(columns=['Date', 'Time', 'datetime'], errors='ignore')

vif_data = pd.DataFrame()
vif_data["Feature"] = X_numeric.columns
vif_data["VIF"] = [variance_inflation_factor(X_numeric.values, i) for i in range(X_numeric.shape[1])]

vif_data

Variance Inflation Factor (VIF) analysis was performed to detect multicollinearity among the predictor variables. The results indicate that Voltage and Global Intensity have high VIF values greater than 10, suggesting strong multicollinearity. This is expected because electrical power is mathematically related to voltage and current. To mitigate this issue, dimensionality reduction using Principal Component Analysis (PCA) will be applied in the modeling stage.

# ***`9. Feature Engineering`***

Additional features were created to improve model performance. Lag features of electricity consumption were generated to capture temporal dependencies in the data. A rolling mean feature was also calculated to represent recent consumption trends. In addition, a new feature called remaining_active_power was derived from the difference between total active power and the sub-metered appliance consumption.



> 9.1 Create a new feature called remaining_active_power



 A new feature called remaining_active_power was derived by subtracting the sub-metered appliance consumption from the total active power. This represents electricity usage from other household appliances not captured by the sub-metering variables.




In [ ]:
df["remaining_active_power"] = (
    df["Global_active_power"] * 1000 / 60
) - (df["Sub_metering_1"] + df["Sub_metering_2"] + df["Sub_metering_3"])



>9.2 Create Lag Feature



Lag features were created to capture temporal dependencies in electricity consumption. These features represent previous consumption values and help the model learn patterns in power usage over time.

In [ ]:
df["lag_1"] = df["Global_active_power"].shift(1)
df["lag_2"] = df["Global_active_power"].shift(2)
df["lag_3"] = df["Global_active_power"].shift(3)

In [ ]:
df = df.dropna()



> 9.3 Rolling Mean Feature



A rolling mean feature was created to capture short-term trends in electricity consumption by calculating the average power usage over a fixed window of previous observations.

In [ ]:
#df["rolling_mean_5"] = df["Global_active_power"].rolling(window=5).mean()

In [ ]:
df.columns

In [ ]:
df.head()

# ***`10. Define Features and Target`***

In [ ]:
X = df.drop(columns=["Global_active_power", "Date", "Time", "datetime"])
y = df["Global_active_power"]

# *`11.Train–Test Split`*

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
print("X_train",X_train.shape)
print("X_test",X_test.shape)
print("y_train",y_train.shape)
print("y_test",y_test.shape)

# ***`12. Feature Scallimg`***

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ***`13.PCA (Dimensionality Reduction)`***

*   PCA can combine features into a new set of uncorrelated features.
* This is particularly useful if the features are linearly related.




In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [ ]:
# After PCA, X_train_pca and X_test_pca are the new feature sets.
# We will now use these PCA-transformed arrays as our training and testing features.

X_train = X_train_pca
X_test = X_test_pca

print("Shape of X_train after PCA:", X_train.shape)
print("Shape of X_test after PCA:", X_test.shape)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA Explained Variance")
plt.show()



The cumulative explained variance plot shows that approximately 6–7 principal components
retain around 95% of the total variance in the dataset. Therefore, PCA with 95% variance
retention was used to reduce dimensionality while preserving most of the information in
the original variables.

# ***`14. Linear Regression Modelling`***

### 14.1 Model Training

A Linear Regression model was trained using the PCA-transformed training data.
The model learns the relationship between the principal components and the
household electricity consumption.


In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

###14.2  Model Prediction

The trained regression model was used to generate predictions on the test dataset.
These predictions represent the estimated electricity consumption based on the
input features.

In [ ]:
y_pred = model.predict(X_test)

### 14.3 Model Evaluation Metrics

To evaluate the performance of the regression model, several evaluation metrics were used.

**1.  Mean Squared Error (MSE)**  
Measures the average of the squared differences between the actual and predicted values.

MSE = (1/n) Σ (yᵢ − ŷᵢ)²


**2.  Mean Absolute Error (MAE)**  
Measures the average absolute difference between actual and predicted values.

MAE = (1/n) Σ |yᵢ − ŷᵢ|


**3.  Root Mean Squared Error (RMSE)**  
Represents the square root of MSE and provides the error in the same unit as the target variable.

RMSE = √( (1/n) Σ (yᵢ − ŷᵢ)² )


**  4.   R² Score (Coefficient of Determination)**  
Indicates how well the model explains the variance in the target variable.

R² = 1 − ( Σ (yᵢ − ŷᵢ)² / Σ (yᵢ − ȳ)² )


**5.   Adjusted R² Score**  
Adjusted R² accounts for the number of predictors used in the model.

Adjusted R² = 1 − [ (1 − R²)(n − 1) / (n − p − 1) ]

where:  
n = number of observations  
p = number of predictors

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# MSE
mse = mean_squared_error(y_test, y_pred)

# MAE
mae = mean_absolute_error(y_test, y_pred)

# RMSE
rmse = np.sqrt(mse)

# R2
r2 = r2_score(y_test, y_pred)

# Adjusted R2
n = len(y_test)
p = X_test_pca.shape[1]

adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

print("Mean Squared Error (MSE):", mse)
print("Mean Absolute Error (MAE):", mae)
print("Root Mean Squared Error (RMSE):", rmse)
print("R2 Score:", r2)
print("Adjusted R2 Score:", adj_r2)

### 14.4 Prediction Visualisation

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("Actual Power Consumption")
plt.ylabel("Predicted Power Consumption")
plt.title("Actual vs Predicted Electricity Consumption")
plt.show()

# ***`15. Residual Analysis `***
### 15.1 Residual Calculation

Residuals represent the difference between the actual and predicted values.
They help evaluate how well the regression model fits the data.

Residual = Actual Value − Predicted Value

In [ ]:
residuals = y_test - y_pred

### 15.2 Residual vs Fitted Values Plot

A residual vs fitted plot is used to examine whether residuals are randomly
distributed around zero. Random distribution indicates that the model
captures the relationship between variables appropriately.

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(y=0, color='red', linestyle='--')

plt.xlabel("Fitted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted Values")

plt.show()

###15.3  Histogram of Residuals

The histogram of residuals helps assess whether the residuals follow a
normal distribution. A roughly bell-shaped distribution suggests that
the regression assumptions are satisfied.

In [ ]:
import seaborn as sns

sns.histplot(residuals, bins=50, kde=True)

plt.title("Histogram of Residuals")
plt.xlabel("Residuals")

plt.show()

### 15.4 Q-Q Plot

A Q-Q plot compares the distribution of residuals with a theoretical
normal distribution. If the points lie approximately along the diagonal
line, the residuals are normally distributed.

In [ ]:
# Ensure residuals are in a 1-dimensional array
residuals = np.ravel(residuals)

# Q-Q Plot
plt.figure(figsize=(10, 6))
stats.probplot(residuals, dist="norm", plot=plt)
plt.title('Q-Q Plot')
plt.show()


### 15.5 Shapiro-Wilk Test

The Shapiro-Wilk test is used to statistically test whether the residuals
follow a normal distribution.

Null Hypothesis (H₀): Residuals are normally distributed.
Alternative Hypothesis (H₁): Residuals are not normally distributed.

If the p-value is greater than 0.05, we fail to reject the null hypothesis,
indicating that the residuals are approximately normally distributed.

In [ ]:
from scipy.stats import shapiro

stat, p_value = shapiro(residuals)

print("Shapiro-Wilk Statistic:", stat)
print("p-value:", p_value)

if p_value > 0.05:
    print("Residuals appear normally distributed")
else:
    print("Residuals are not normally distributed")

### 15.6 Cross Validation

Cross-validation is used to evaluate the model's performance by dividing
the dataset into multiple training and validation subsets. This helps
assess how well the model generalizes to unseen data.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

model = LinearRegression()

scores = cross_val_score(model, X_train_pca, y_train, cv=5, scoring='r2')

print("Cross Validation R2 Scores:", scores)
print("Average CV Score:", scores.mean())

# ***`Model Interpretation`***

The regression model was able to capture the general relationship between the predictor variables and household electricity consumption. The actual versus predicted plot shows that the predictions follow the overall trend of the observed values, indicating that the model is able to learn meaningful patterns from the data.

Residual analysis suggests that while the residuals are not perfectly normally distributed, this is expected due to the large sample size and real-world variability in electricity usage. Overall, the model provides reasonable predictions of electricity consumption based on the available electrical and temporal features.

# ***`Conclusion`***

This project developed a machine learning regression model to predict
household electricity consumption using historical electrical and
temporal features.

The analysis included data cleaning, exploratory data analysis,
feature engineering, multicollinearity detection, dimensionality
reduction using PCA, and model evaluation.

The results show that the model is capable of predicting electricity
consumption patterns with reasonable accuracy. Such models can be
useful for energy management, demand forecasting, and smart grid
applications.